# VirtualiZarr → Icechunk in Cloud Storage → Append days
### Author: Eli Holmes modification of Rich Signell's notebook
[![Colab Badge](https://img.shields.io/badge/Open_in_Colab-blue?style=for-the-badge)][colab-link] [![Download Badge](https://img.shields.io/badge/Download-grey?style=for-the-badge)][download-link] [![JupyterHub](https://img.shields.io/badge/Jupyter_Hub-orange?style=for-the-badge)][jupyter-link]

[download-link]: https://github.com/nmfs-opensci/nmfshackdays-2026/blob/main/topics/2026-06-05/virtualizarr_ndvi_cdr_append-cloud.ipynb
[colab-link]: https://colab.research.google.com/github/nmfs-opensci/nmfshackdays-2026/blob/main/topics/2026-06-05/virtualizarr_ndvi_cdr_append-cloud.ipynb
[jupyter-link]: https://nmfs-openscapes.2i2c.cloud/hub/user-redirect/lab?fromURL=https://raw.githubusercontent.com/nmfs-opensci/nmfshackdays-2026/main/topics/2026-06-05/virtualizarr_ndvi_cdr_append-cloud.ipynb

##  Workflow

1. Open a single nc file on S3 and create a virtual dataset
2. Write virtual references to an Icechunk store in cloud storage
3. Open the next nc file on S3 and create a virtual dataset
4. Append to the Icechunk store
5. Repeat

In [ ]:
# Colab users, uncomment and run this
#!pip install -q icechunk  virtualizarr xarray obspec_utils obstore hvplot s3fs

In [1]:
import warnings
import shutil
from pathlib import Path

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset, open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## Set up the info on my buckets

In [48]:
# Where the data are
source_data_bucket = "s3://noaa-oar-cefi-regional-mom6-pds"
source_data_prefix = "northeast_pacific/full_domain/hindcast/daily/regrid/r20250912"
source_data_region = "us-east-1"

# Where my icechunk is
icechunk_bucket = "us-west-2.opendata.source.coop"
icechunk_prefix = "eeholmes/cefi/nepacific-icechunk"
icechunk_region = "us-west-2"

# Where my creds for my icechunk bucket are
icechunk_creds = "source-cefi-creds.json"

In [56]:
#Here is how you can figure out region
# Figure out the region
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED),
)
# leave off the s3:// part
s3.head_bucket(Bucket="noaa-oar-cefi-regional-mom6-pds")

{'ResponseMetadata': {'RequestId': '44JQ34M8FT7QQQG3',
  'HostId': '3mxC+Zk5bHYexGaO84Symw/ZMuNfWhJb4VMfo/3RNvxH+LvPFL0CPnlU2sVAb+32nRs1hfl5FbJHuze7TYe/qvGsoL6o4CE4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '3mxC+Zk5bHYexGaO84Symw/ZMuNfWhJb4VMfo/3RNvxH+LvPFL0CPnlU2sVAb+32nRs1hfl5FbJHuze7TYe/qvGsoL6o4CE4',
   'x-amz-request-id': '44JQ34M8FT7QQQG3',
   'date': 'Fri, 05 Jun 2026 18:16:18 GMT',
   'x-amz-bucket-region': 'us-east-1',
   'x-amz-access-point-alias': 'false',
   'x-amz-bucket-arn': 'arn:aws:s3:::noaa-oar-cefi-regional-mom6-pds',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 1},
 'BucketArn': 'arn:aws:s3:::noaa-oar-cefi-regional-mom6-pds',
 'BucketRegion': 'us-east-1',
 'AccessPointAlias': False}

## Set up the urls to the REMOTE data files in object storage

Start by looking at what is in the bucket and seeing how the data are organized.

In [50]:
import s3fs
# s3fs since this is a s3 bucket
# This is a public bucket so anon=True
fs = s3fs.S3FileSystem(anon=True)
fs.ls(source_data_bucket)

['noaa-oar-cefi-regional-mom6-pds/index.html',
 'noaa-oar-cefi-regional-mom6-pds/northeast_pacific',
 'noaa-oar-cefi-regional-mom6-pds/northwest_atlantic']

In [51]:
# Let's look at what is in the regrid data
fs.ls(f"{source_data_bucket}/northeast_pacific/full_domain/hindcast/")

['noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily',
 'noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/monthly']

In [52]:
# Let's look at what is in data; looks like daily based on file names
fs.ls(f"{source_data_bucket}/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912")[:5]

['noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/btm_co3_ion.nep.full.hcast.daily.regrid.r20250912.199301-202506.json',
 'noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/btm_co3_ion.nep.full.hcast.daily.regrid.r20250912.199301-202506.nc',
 'noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/btm_co3_sol_arag.nep.full.hcast.daily.regrid.r20250912.199301-202506.json',
 'noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/btm_co3_sol_arag.nep.full.hcast.daily.regrid.r20250912.199301-202506.nc',
 'noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/btm_co3_sol_calc.nep.full.hcast.daily.regrid.r20250912.199301-202506.json']

In [53]:
# Create urls for chlos. Files are yearly
http_prefix = "s3://"
prefix = "northeast_pacific/full_domain/hindcast/daily/regrid/r20250912"

# sort to ensure time ordered
filenames = sorted(
    f for f in fs.ls(f"{source_data_bucket}/{prefix}")
    if f.endswith(".nc") and Path(f).name.startswith("chlos.")
)

urls = [ f if f.startswith("s3://") else f"{http_prefix}{f}" for f in filenames]

urls[:3]

['s3://noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/chlos.nep.full.hcast.daily.regrid.r20250912.199301-199312.nc',
 's3://noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/chlos.nep.full.hcast.daily.regrid.r20250912.199401-199412.nc',
 's3://noaa-oar-cefi-regional-mom6-pds/northeast_pacific/full_domain/hindcast/daily/regrid/r20250912/chlos.nep.full.hcast.daily.regrid.r20250912.199501-199512.nc']

In [54]:
test_url = urls[0]

ds0 = xr.open_dataset(
    test_url,
    engine="h5netcdf",
    backend_kwargs={
        "storage_options": {
            "anon": True,
            "client_kwargs": {"region_name": "us-east-1"},
        }
    },
)

ds0

<xarray.Dataset> Size: 406MB
Dimensions:  (time: 365, lat: 815, lon: 341)
Coordinates:
  * time     (time) datetime64[ns] 3kB 1993-01-01T12:00:00 ... 1993-12-31T12:...
  * lat      (lat) float64 7kB 10.81 10.89 10.98 11.07 ... 80.55 80.63 80.72
  * lon      (lon) float64 3kB 156.9 157.2 157.5 157.8 ... 254.4 254.7 255.0
Data variables:
    chlos    (time, lat, lon) float32 406MB ...
Attributes: (12/27)
    NumFilesInSet:          1
    title:                  NEP10k_202507_physics_bgc
    associated_files:       areacello: 19930101.ocean_static.nc
    grid_type:              regular
    grid_tile:              N/A
    external_variables:     areacello
    ...                     ...
    cefi_init_date:         N/A
    cefi_ensemble_info:     N/A
    cefi_forcing:           N/A
    cefi_data_doi:          10.5281/zenodo.13936240
    cefi_paper_doi:         10.5194/gmd-2024-195
    cefi_aux:               Postprocessed Data : regrid to regular grid

## Setup S3 store, registry and Icechunk config for REMOTE files

Point `obstore` at the public NDVI bucket and register it so VirtualiZarr can resolve chunk references.

In [68]:
# Create an object-store handle for the REMOTE files.
store = from_url(bucket, region=source_data_region, skip_signature=True)
registry = ObjectStoreRegistry({bucket: store})
parser = HDFParser()

This will be passed to `icechunk` functions.

In [69]:
# Tell the store how to access the remote chunks
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{source_data_bucket}/", # need that trailing /
        store=icechunk.s3_store(region=source_region, anonymous=True),
    ),
)

### Set up the Icechunk storage bucket

Source Coop makes it easy to get this info via a 'View Credentials' link. Since I am working in a Jupyter notebook, I will load the variables via a json file. 

1. Create a json file and add to `.gitignore` so you do not accidentally commit it to GitHub. Never hard-code tokens into notebooks.
2. Get the bucket info: provider (S3, GCP, etc), bucket, prefix and pass that to set up the storage object.

![](sc_env_var.png)

In [70]:
# Read in the json file
import json
from pathlib import Path

with open(icechunk_creds) as f:
    source_creds = json.load(f)

In [71]:
# This uses info on the bucket where the icechunk store is
storage = icechunk.s3_storage(
    bucket=icechunk_bucket,
    prefix=icechunk_prefix,
    region=source_creds["region_name"],
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
)

In [79]:
# Create store if it is empty
try:
    repo = icechunk.Repository.create(storage, config)
    print("Created new Icechunk repo")
except Exception:
    repo = icechunk.Repository.open(storage, config=config)
    print("Opened existing Icechunk repo")

# Create a session
session = repo.writable_session(branch="main")

Created new Icechunk repo


## Once set up, the rest is the same

In [80]:
# Run the for loop
import time
from pathlib import Path
loadable_variables=["time", "lat", "lon"]

for i, url in enumerate(urls[:5]):
    filename = Path(url).name
    start = time.perf_counter()

    print(f"Adding {filename}")

    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=loadable_variables,
        decode_times=True,
    ).drop_vars(["nv", "ncrs", "crs"], errors="ignore")

    if i == 0:
        vds.vz.to_icechunk(session.store)
    else:
        vds.vz.to_icechunk(session.store, append_dim="time")

    elapsed = time.perf_counter() - start
    print(f"Finished {filename} in {elapsed:.2f} seconds")

snapshot_id = session.commit("1 year")
print("Committed:", snapshot_id)

Adding chlos.nep.full.hcast.daily.regrid.r20250912.199301-199312.nc
Finished chlos.nep.full.hcast.daily.regrid.r20250912.199301-199312.nc in 5.15 seconds
Adding chlos.nep.full.hcast.daily.regrid.r20250912.199401-199412.nc
Finished chlos.nep.full.hcast.daily.regrid.r20250912.199401-199412.nc in 4.74 seconds
Adding chlos.nep.full.hcast.daily.regrid.r20250912.199501-199512.nc
Finished chlos.nep.full.hcast.daily.regrid.r20250912.199501-199512.nc in 4.89 seconds
Adding chlos.nep.full.hcast.daily.regrid.r20250912.199601-199612.nc
Finished chlos.nep.full.hcast.daily.regrid.r20250912.199601-199612.nc in 4.88 seconds
Adding chlos.nep.full.hcast.daily.regrid.r20250912.199701-199712.nc
Finished chlos.nep.full.hcast.daily.regrid.r20250912.199701-199712.nc in 4.54 seconds
Committed: TDKD5K447SYH7J00TM4G


In [81]:
# So about 5 minutes to do all 40 years
# If done in series, not parallel
40*4.5/(60*60)

0.05

## Public reading of our icechunk

Add these as instructions in for your dataset.

In [82]:
import icechunk
import xarray as xr

# -------------------------------------------------------------------
# 1. Authorize access to the original NOAA NetCDF chunks
# -------------------------------------------------------------------
# The Icechunk repo contains virtual chunk references back to a
# public NOAA S3 bucket. 
credentials = icechunk.containers_credentials({
    source_data_bucket: icechunk.s3_credentials(anonymous=True)
})

# -------------------------------------------------------------------
# 2. Tell Icechunk where virtual chunks are allowed to come from
# -------------------------------------------------------------------
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{source_data_bucket}/", # need that trailing /
        store=icechunk.s3_store(region=source_data_region, anonymous=True),
    ),
)

# -------------------------------------------------------------------
# 3. Point to the public Icechunk repo on Source Cooperative
# -------------------------------------------------------------------
# This is the location of the Icechunk repository itself.
storage = icechunk.s3_storage(
    bucket=icechunk_bucket,
    prefix=icechunk_prefix,
    region=icechunk_region,
    anonymous=True,
)

# -------------------------------------------------------------------
# 4. Open the Icechunk repo and read it with xarray
# -------------------------------------------------------------------
repo = icechunk.Repository.open(
    storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)

session = repo.readonly_session(branch="main")

ds = xr.open_zarr(
    session.store,
    consolidated=False,
    chunks=None,
)

ds

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 1826, lat: 815, lon: 341)
Coordinates:
  * time     (time) datetime64[ns] 15kB 1993-01-01T12:00:00 ... 1997-12-31T12...
  * lat      (lat) float64 7kB 10.81 10.89 10.98 11.07 ... 80.55 80.63 80.72
  * lon      (lon) float64 3kB 156.9 157.2 157.5 157.8 ... 254.4 254.7 255.0
Data variables:
    chlos    (time, lat, lon) float32 2GB ...
Attributes: (12/27)
    NumFilesInSet:          1
    title:                  NEP10k_202507_physics_bgc
    associated_files:       areacello: 19970101.ocean_static.nc
    grid_type:              regular
    grid_tile:              N/A
    external_variables:     areacello
    ...                     ...
    cefi_init_date:         N/A
    cefi_ensemble_info:     N/A
    cefi_forcing:           N/A
    cefi_data_doi:          10.5281/zenodo.13936240
    cefi_paper_doi:         10.5194/gmd-2024-195
    cefi_aux:               Postprocessed Data : regrid to regular grid

## We can vizualize Icechunks in public stores

Javascript libraries `zarrita` and `icechunk-js` allow us to create visualizations of public Zarr stores.

https://eeholmes.github.io/gridlook/#https://data.source.coop/eeholmes/cefi/nepacific-icechunk


## Make a plot with hvplot

In [66]:
import hvplot.xarray

ds["chlos"].isel(time=0).hvplot.quadmesh(
    rasterize=True,
    x="lon",
    y="lat",
    title="chlos",
    width=800,
    height=400,
    cmap = "turbo_r"
)

:DynamicMap   []
   :Image   [lon,lat]   (Surface Mass Concentration of Total Phytoplankton expressed as Chlorophyll in Sea Water)

## Troubleshooting

If you mess up and need to remove the icechunk store....

In [ ]:
%%script false --no-raise
# remove above if you want to run this cell

import json
import boto3

# Icechunk repo location on Source Cooperative
icechunk_bucket = "us-west-2.opendata.source.coop"
icechunk_prefix = "eeholmes/cefi/nepacific-icechunk"  # delete everything under this prefix

# Load temporary Source upload credentials
with open("source-cefi-creds.json") as f:
    source_creds = json.load(f)

icechunk_region = "us-west-2"

s3 = boto3.client(
    "s3",
    region_name=icechunk_region,
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds["aws_session_token"],
)

def delete_s3_prefix(bucket, prefix):
    prefix = prefix.strip("/") + "/"

    paginator = s3.get_paginator("list_objects_v2")
    total = 0

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        objects = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]

        if objects:
            s3.delete_objects(
                Bucket=bucket,
                Delete={"Objects": objects},
            )
            total += len(objects)

    print(f"Deleted {total} objects from s3://{bucket}/{prefix}")

delete_s3_prefix(icechunk_bucket, icechunk_prefix)